# The Four Pillars of OOP

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Protect an object's state with private fields and methods that check every change (encapsulation).
- Build a class on top of another with extends and super, and override its methods (inheritance).
- Treat objects of different classes through their parent type, each answering in its own way (polymorphism).

**In this notebook**, one idea at a time (run each cell before moving on):
1. The four pillars
2. Encapsulation: private fields
3. Inheritance: extends and super
4. Overriding and polymorphism
5. Abstraction
6. Common mistakes

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## 1. The four pillars

Object-oriented programming rests on four ideas. You've already used the first two a little; this lesson goes through each one.

| Pillar | Means | In Java |
|---|---|---|
| **Encapsulation** | an object protects its own state | `private` fields, changed only through methods |
| **Inheritance** | a class builds on another | `class Dog extends Animal` |
| **Polymorphism** | one call, a different answer for each kind of object | overriding methods, a parent-type variable |
| **Abstraction** | code works with the general idea, not each kind's details | a method or type that hides how things are done |

## 2. Encapsulation: private fields

If any code can change a field, nothing stops an invalid value, such as a thermostat at 500 degrees. Mark the fields `private`: only the class's own methods can touch them. Other code goes through methods that check every change. A method that only reads a field is often called a **getter**.

In [ ]:
class Thermostat {
    private int target = 20;

    void setTarget(int degrees) {
        if (degrees < 10 || degrees > 30) {
            IO.println("Ignored: " + degrees + " is out of range");
            return;
        }
        target = degrees;
    }

    int getTarget() {
        return target;
    }
}

void main() {
    Thermostat t = new Thermostat();
    t.setTarget(23);
    t.setTarget(500);
    IO.println("Target: " + t.getTarget());
}

🧪 **Try it:** in `main`, write `t.target = 500;` and read the error: `private` stops it. Then add `void warmer()`, which raises the target by 1 but never above 30.

## 3. Inheritance: extends and super

`class Dog extends Animal` makes Dog a **subclass** of Animal: it inherits Animal's fields and methods, and can add its own. Its constructor must first build the Animal part, with `super(...)` as its first line. The parent's `private` fields stay private: the subclass reaches them through the parent's methods.

In [ ]:
class Animal {
    private final String name;

    Animal(String name) {
        this.name = name;
    }

    String name() {
        return name;
    }
}

class Dog extends Animal {
    private final String breed;

    Dog(String name, String breed) {
        super(name);              // builds the Animal part first
        this.breed = breed;
    }

    String card() {
        return name() + ", a " + breed;   // name() is inherited
    }
}

void main() {
    Dog rex = new Dog("Rex", "beagle");
    IO.println(rex.card());
    IO.println(rex.name());
}

🧪 **Try it:** remove the `super(name);` line and read the error. Then, in `card()`, use the field `name` directly instead of `name()`: it's private to Animal.

### ✏️ Exercise: A counter that counts in steps

Counter adds 1. Make StepCounter override `add()` so it adds `step`, by calling the parent's `add()` with `super.add()` that many times.

In [ ]:
class Counter {
    private int value;

    void add() {
        value++;
    }

    int value() {
        return value;
    }
}

class StepCounter extends Counter {
    private final int step;

    StepCounter(int step) {
        this.step = step;
    }

    @Override
    void add() {
        // TODO: call the parent's add() `step` times, with super.add()
    }
}

void main() {
    StepCounter three = new StepCounter(3);
    three.add();
    three.add();
    check("two adds of 3", three.value(), 6);
    StepCounter one = new StepCounter(1);
    one.add();
    check("a step of 1", one.value(), 1);
}

<details>
<summary>Show an answer</summary>

```java
class Counter {
    private int value;

    void add() {
        value++;
    }

    int value() {
        return value;
    }
}

class StepCounter extends Counter {
    private final int step;

    StepCounter(int step) {
        this.step = step;
    }

    @Override
    void add() {
        for (int i = 0; i < step; i++) {
            super.add();
        }
    }
}

void main() {
    StepCounter three = new StepCounter(3);
    three.add();
    three.add();
    check("two adds of 3", three.value(), 6);
    StepCounter one = new StepCounter(1);
    one.add();
    check("a step of 1", one.value(), 1);
}
```

</details>

## 4. Overriding and polymorphism

A subclass can **override** a method: write a method with the same name and parameters, marked `@Override`. Then a variable of the parent type can hold any subclass, and calling the method runs the version of the **object's** class, chosen while the program runs. That's **polymorphism**: one line of code, a different answer for each kind of object. The slides follow it.

#### 🎞️ Which sound() runs?

![A List<Animal> holds a Dog and an Animal; describe() is inherited from Animal and calls sound(), and Java runs the version of each object's own class: Woof for the Dog, ... for the Animal](dispatch-steps.svg)

In [ ]:
class Animal {
    private final String name;

    Animal(String name) {
        this.name = name;
    }

    String describe() {
        return name + ": " + sound();
    }

    String sound() {
        return "...";
    }
}

class Dog extends Animal {
    Dog(String name) {
        super(name);
    }

    @Override
    String sound() {
        return "Woof";
    }
}

class Cat extends Animal {
    Cat(String name) {
        super(name);
    }

    @Override
    String sound() {
        return "Meow";
    }
}

void main() {
    List<Animal> zoo = List.of(new Dog("Rex"), new Cat("Mia"), new Animal("Ann"));
    for (Animal a : zoo) {
        IO.println(a.describe());
    }
}

🧪 **Try it:** add a `Cow` and put it in the list: the loop needs no change. Then misspell `sound` in Cat as `Sound`, keeping `@Override`, and read the error.

### ✏️ Exercise: Wheels

Make Bike override `wheels()` to return 2, and write `totalWheels`, which adds up the wheels of every vehicle in the list. The loop calls `wheels()` without knowing which class each vehicle is.

In [ ]:
class Vehicle {
    int wheels() {
        return 4;
    }
}

class Bike extends Vehicle {
    // TODO: override wheels() to return 2
}

int totalWheels(List<Vehicle> vehicles) {
    // TODO: add up v.wheels() for every vehicle
    return 0;
}

void main() {
    check("a bike", new Bike().wheels(), 2);
    check("a car and two bikes", totalWheels(List.of(new Vehicle(), new Bike(), new Bike())), 8);
    check("no vehicles", totalWheels(List.of()), 0);
}

<details>
<summary>Show an answer</summary>

```java
class Vehicle {
    int wheels() {
        return 4;
    }
}

class Bike extends Vehicle {
    @Override
    int wheels() {
        return 2;
    }
}

int totalWheels(List<Vehicle> vehicles) {
    int total = 0;
    for (Vehicle v : vehicles) {
        total += v.wheels();
    }
    return total;
}

void main() {
    check("a bike", new Bike().wheels(), 2);
    check("a car and two bikes", totalWheels(List.of(new Vehicle(), new Bike(), new Bike())), 8);
    check("no vehicles", totalWheels(List.of()), 0);
}
```

</details>

## 5. Abstraction

**Abstraction** means code works with the general idea and leaves the details to each kind. `describe()` above never asks "is it a dog?": it just calls `sound()`. A method that receives an `Animal` works for every animal, even kinds written later. The next lesson takes this further: an **abstract class** or an **interface** describes the idea without any default version at all.

In [ ]:
class Animal {
    String sound() {
        return "...";
    }
}

class Dog extends Animal {
    @Override
    String sound() {
        return "Woof";
    }
}

String chorus(List<Animal> animals) {       // knows only the general idea
    StringBuilder sb = new StringBuilder();
    for (Animal a : animals) {
        sb.append(a.sound()).append(' ');
    }
    return sb.toString().trim();
}

void main() {
    IO.println(chorus(List.of(new Dog(), new Dog(), new Animal())));
}

🧪 **Try it:** write a `Duck` subclass and use it in `chorus` without touching `chorus` itself.

## 6. Common mistakes

- **A typo instead of an override:** without `@Override`, `String Sound()` is a new method, and the parent's `sound()` keeps running.
- **`super(...)` not first:** the parent's part must be built before anything else in the constructor.
- **Reaching a parent's `private` field:** use the parent's methods instead.
- **Calling a subclass-only method through the parent type:** an `Animal` variable only offers Animal's methods, even when it holds a Dog.

In [ ]:
class Animal {
    String sound() {
        return "...";
    }
}

class Dog extends Animal {
    String Sound() {              // a typo: this doesn't replace sound()
        return "Woof";
    }

    void fetch() {
        IO.println("fetching!");
    }
}

void main() {
    Animal a = new Dog();
    IO.println(a.sound());        // still "...": nothing overrode it
    Dog d = new Dog();
    d.fetch();
}

🧪 **Try it:** put `@Override` above `String Sound()` and read the error. Fix the name. Then try `a.fetch();`: an `Animal` variable doesn't offer it.

## In short

OOP has four pillars:

| Pillar | Means |
|---|---|
| **Encapsulation** | `private` fields, changed only through methods |
| **Inheritance** | `class Dog extends Animal` reuses and extends a parent class |
| **Polymorphism** | an `Animal` variable holds any subclass, each answering its own way |
| **Abstraction** | code uses the general idea (`Animal`), not each kind's details |

```java
class Animal {
    private final String name;

Animal(String name) {
        this.name = name;
    }

String describe() {
        return name + ": " + sound();
    }

String sound() {
        return "...";
    }
}

class Dog extends Animal {
    Dog(String name) {
        super(name);
    }

@Override
    String sound() {
        return "Woof";
    }
}

void main() {
    List<Animal> zoo = List.of(new Dog("Rex"), new Animal("Generic"));
    for (Animal a : zoo) {
        IO.println(a.describe());
    }
}
```

`describe()` calls `sound()`, and Java picks the version of the **object's** class while the program runs.

**Watch out:** `@Override` asks the compiler to check that you really replace a parent's method. Without it, a typo such as `Sound()` silently creates a new method.

## Readings

- [Polymorphism](https://dev.java/learn/language/oop/inheritance/polymorphism/) (dev.java (Oracle), 7 min): The Bicycle example: each subclass overrides printDescription, and the same call gives a different answer. Then super for calling the parent's version, and how subclass constructors call the parent's.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Bank Account (Encapsulation)** · Medium
- **Shapes (Inheritance)** · Medium
- **Payroll (Polymorphism)** · Hard